# E66 — think 비용 예측용 추론모델 prior

**런타임 → A100 권장 (L4도 7B면 가능). 게이트까지 ~1.5시간.**

## 왜 이걸 하는가

E64에서 남은 여지의 소재가 정확히 밝혀졌다. **참비용으로 할당하면 어떤 안전계수에서도 초과가 0%다.**
즉 지금 묶여 있는 안전 마진 전부가 **비용 예측 오차에 대한 보험**이고, 그 오차는 think 헤드에 몰려 있다:

| 헤드 | 로그비용 RMSE |
|---|---:|
| ax31-light | 0.556 |
| ax31 | 0.458 |
| **axk1-think** | **0.677** |

이걸 없애면 **약 +0.022**(fast +0.0053 / balanced +0.0067 / premium +0.0104 가중)로, 34B 컬럼 작업의
3배다. 남은 유일한 큰 축이다.

그런데 아티팩트 안에 이 신호가 없다. think 비용의 87%가 출력 토큰인데,

| 예측자 | think 로그출력길이와의 corr |
|---|---:|
| 34B 컬럼(C)의 출력길이 | 0.182 |
| **실제 `ax31`의 출력길이** | **0.319** ← 현재 최고 |

**추론 모델은 think처럼 긴 사고 사슬을 뿜는 유일한 대리자다.** E57이 추론 컬럼을 안 만든 이유는
*점수* 신호가 이미 중복이라서였는데, 지금 필요한 건 점수가 아니라 **비용**이다. 이건 아무도 안 재봤다.

**게이트: `corr(추론모델 출력길이, think 출력길이) ≥ 0.45`.** 0.319를 유의미하게 넘어야 의미가 있다.
공개 2,640문항만 먼저 돌려 1시간 안에 판정하고, 통과할 때만 대량 라벨링으로 넘어간다.

In [ ]:
#@title ① 번들 압축 해제
import os, zipfile, glob, shutil
BUNDLE = 'e66_colab_bundle.zip'
if not os.path.exists(BUNDLE):
    try:
        from google.colab import drive; drive.mount('/content/drive')
        c = glob.glob('/content/drive/MyDrive/**/' + BUNDLE, recursive=True)
        if c: shutil.copyfile(c[0], BUNDLE)
    except Exception as e: print('drive skip:', e)
if not os.path.exists(BUNDLE) or not zipfile.is_zipfile(BUNDLE):
    from google.colab import files; files.upload()
print('zip 정상:', zipfile.is_zipfile(BUNDLE), f'{os.path.getsize(BUNDLE)/1e6:.1f} MB')
with zipfile.ZipFile(BUNDLE) as z: z.extractall('.')
%cd /content/official-router
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title ② 의존성 + 설정
!pip -q install vllm datasets bitsandbytes 2>&1 | tail -2
!pip -q uninstall -y torchaudio 2>&1 | tail -1
import os, torch, transformers, vllm
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-14B'  #@param ['deepseek-ai/DeepSeek-R1-Distill-Qwen-14B', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-32B']
os.environ.update(
    MODEL=MODEL,
    N='2',            # 길이 추정이 목적이라 2회면 충분 (점수용이 아니다)
    TEMP='0.6',       # R1 계열 권장 온도
    MAXTOK='4096',    # 사고 사슬을 자르면 길이 신호 자체가 사라진다 -- 여기서 아끼면 안 된다
    MAXLEN='16384',
    QUANT='' if GB >= 70 else 'bitsandbytes',
    UTIL='0.92',
    TOKENIZERS_PARALLELISM='false')
print('OK', torch.__version__, f'{GB:.0f} GB', MODEL,
      'quant=' + (os.environ['QUANT'] or 'bf16'))

**`--instruct`를 쓰지 않는다.** 주최측 형식 지시문은 답을 짧게 뽑아내려고 붙이는 것인데, 여기서 재는 건
정답이 아니라 **모델이 이 문제에 얼마나 오래 생각하는가**다. 지시문을 붙이면 그 신호가 눌린다.

In [ ]:
#@title ③ 게이트용 라벨링 — 공개 2,640문항 (~1시간, 사고 사슬이 길어 토큰이 많다)
import os
os.makedirs('colab-label/out', exist_ok=True)
!PYTHONPATH=src python -X utf8 colab-label/run_labels.py --stage pilot \
    --model "$MODEL" --engine vllm --n $N --temps $TEMP \
    --max-model-len $MAXLEN --max-tokens $MAXTOK --gpu-util $UTIL \
    ${QUANT:+--quant $QUANT} --tag _reason \
    --bundle colab-label/bundle --out colab-label/out 2>&1 | grep -v -i warn | tail -25
!wc -l colab-label/out/labels_pilot_T${TEMP}_reason.jsonl

In [ ]:
#@title ④ 게이트 — corr(추론모델 길이, think 길이) 가 0.319 를 넘는가
import glob
hits = sorted(glob.glob('colab-label/out/labels_pilot*_reason.jsonl'))
if not hits:
    print('라벨이 없다 -> ③의 오류를 먼저 확인할 것')
else:
    print('using', hits[-1])
    !PYTHONPATH=src python -X utf8 colab-label/think_cost_gate.py \
        --labels {hits[-1]} --items colab-label/bundle/pilot.jsonl 2>&1 | tail -30

**④ 판정** — 마지막 `GATE:` 줄.
- **≥ 0.45** → 통과. ⑤로 가서 대량 라벨링(3~5시간). 그 다음 로컬에서 비용 헤드에 넣고 안전계수를 재가격한다.
- **< 0.45** → 여기서 종료. GPU 시간을 더 쓰지 말고 표를 그대로 붙여넣을 것. 이 경우 결론은
  "think의 출력 길이는 어떤 대리 모델로도 예측 불가"이고, 그러면 안전 마진은 줄일 수 없는 것이 되어
  **0.7019가 이 구조의 천장**이라는 뜻이 된다.
- `추론모델 + ax31 결합` 행도 같이 볼 것. 단독으로는 0.319에 못 미쳐도 결합이 0.45를 넘으면 가치가 있다.

In [ ]:
#@title ⑤ (게이트 통과 시) 대량 라벨링 — 재개 지원
!PYTHONPATH=src python -X utf8 colab-label/run_labels.py --stage pool \
    --model "$MODEL" --engine vllm --n $N --temp $TEMP \
    --max-model-len $MAXLEN --max-tokens $MAXTOK --gpu-util $UTIL \
    ${QUANT:+--quant $QUANT} --tag _reason \
    --bundle colab-label/bundle --out colab-label/out 2>&1 | grep -v -i warn | tail -30
# 중간 저장(권장): !cp colab-label/out/*_reason.jsonl /content/drive/MyDrive/

In [ ]:
#@title ⑥ 결과 회수 → MyDrive
!cd /content/official-router && zip -qr /content/e66_out.zip colab-label/out/*_reason.jsonl
!ls -la /content/e66_out.zip
import zipfile, os
print('zip 정상:', zipfile.is_zipfile('/content/e66_out.zip'))
from google.colab import drive; drive.mount('/content/drive')
!cp /content/e66_out.zip /content/drive/MyDrive/
s, d = '/content/e66_out.zip', '/content/drive/MyDrive/e66_out.zip'
print('MyDrive 사본 크기 일치:', os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s))